# FedFlow-LLM: Colab training run (Telecom Italia, Milan data)

Run cells top to bottom, one at a time. Never run a cell twice.

Use a GPU runtime (T4).

**Keep-alive (optional, not a cell):** press F12, open the Console tab on this Colab page, and paste this snippet so Colab doesn't disconnect while it's idle:

```javascript
function keepAlive() {
  // Click the "Connect"/status button every 60 s so Colab sees activity.
  const btn = document.querySelector("colab-connect-button");
  if (btn) btn.shadowRoot.querySelector("#connect").click();
}
setInterval(keepAlive, 60000);
```

**What differs from fedflow_colab.ipynb:** the data comes from `datasets/telecom_milan/cells.csv` (six cells, one per client) instead of the synthetic series. This needs `datasets/telecom_milan/` to be on the GitHub branch that the clone cell downloads, so push it first. Training code is unchanged; the longest telecom training example is 256 tokens (`datasets/telecom_milan/token_audit.txt`), which fits `client.py`'s default limit of 256.

In [ ]:
# Confirm a GPU is attached.
!nvidia-smi

In [ ]:
# Install dependencies (order matters, see README).
!pip install -q torch transformers peft accelerate bitsandbytes "flwr[simulation]"
!pip uninstall -y torch
!pip install -q torch --index-url https://download.pytorch.org/whl/cu121
!pip uninstall -y torchvision torchaudio
!pip install -q transformers==4.46.3 peft==0.13.2

In [ ]:
# Clone the repo and move into training/, where every script runs.
!git clone https://github.com/shashank69420/fedflow-llm.git
%cd fedflow-llm/training

In [ ]:
# Generate the telecom data (into data/telecom/) and check that training, validation and test windows share no slot,
# and that each client's slice holds only its own cell.
!python generate_data.py --dataset telecom && python check_split.py --dataset telecom

client.py reads `data/traffic_train.json`, and infer.py reads `data/test_windows.json`. The next cell copies these four telecom files from `data/telecom/` into `data/`:

- `traffic_train.json` (200 examples; example j belongs to cell j % 6, so client i trains on cell i)
- `train_starts.json`
- `val_windows.json`
- `test_windows.json`

This is a fresh clone, so no synthetic file is overwritten.

In [ ]:
# Copy the four telecom files into data/, where client.py and infer.py look, and confirm each copy is identical.
import filecmp
import shutil

for name in ["traffic_train.json", "train_starts.json", "val_windows.json", "test_windows.json"]:
    shutil.copyfile(f"data/telecom/{name}", f"data/{name}")
    assert filecmp.cmp(f"data/telecom/{name}", f"data/{name}", shallow=False), f"copy of {name} differs"
    print("copied", name)

!ls -l data/

SMOKE TEST: run the next cell, then STOP and report the output.

In [ ]:
# Smoke test: 1 training round, then inference on 5 held-out windows.
!FEDFLOW_ROUNDS=1 python server.py && python infer.py --eval --limit 5

In [ ]:
# Show the smoke-test output files and the metrics.
!ls -lh outputs
!cat outputs/metrics.json

DO NOT RUN BELOW until the smoke test has been checked.

In [ ]:
# Full run: 25 training rounds, then inference on all held-out windows.
!FEDFLOW_ROUNDS=25 python server.py && python infer.py --eval

In [ ]:
# Zip outputs/ and download it. In the repo, unzip it into results/final_telecom/ (so results/final_telecom/outputs/forecasts.csv exists).
!zip -r outputs.zip outputs/
from google.colab import files
files.download("outputs.zip")

# VALIDATION RUN OF THE v1 MODEL (no training)

Scores the saved v1 adapter on the validation windows. Nothing is trained here, and the test outputs are not overwritten (this writes `outputs/val_forecasts.csv` and `outputs/val_metrics.json`).

Use a fresh GPU runtime. Run only the setup cells at the top (GPU check, install, clone, data generation, data copy), skip every training cell, then run the three cells below. The copy cell must have run, because `infer.py --split val` reads `data/val_windows.json`.

In [ ]:
# Upload the outputs.zip saved after the full training run and put its adapter at outputs/adapter.npz.
import glob
import os
import shutil
import zipfile
from google.colab import files

uploaded = files.upload()  # choose outputs.zip in the dialog
zip_name = next(iter(uploaded))  # Colab may rename a repeat upload, e.g. "outputs (1).zip"

# Extract to a scratch folder first: the zip may or may not contain a top-level outputs/ folder.
shutil.rmtree("unzipped_v1", ignore_errors=True)
with zipfile.ZipFile(zip_name) as z:
    z.extractall("unzipped_v1")
found = glob.glob("unzipped_v1/**/adapter.npz", recursive=True)
assert len(found) == 1, f"expected exactly one adapter.npz in the zip, found {found}"

os.makedirs("outputs", exist_ok=True)
shutil.copyfile(found[0], "outputs/adapter.npz")
assert os.path.exists("outputs/adapter.npz"), "outputs/adapter.npz is missing"
print("adapter ready:", os.path.getsize("outputs/adapter.npz"), "bytes")

In [ ]:
# Score the validation windows with the uploaded adapter (writes outputs/val_forecasts.csv and outputs/val_metrics.json).
!python infer.py --eval --split val

In [ ]:
# Zip the two validation files and download them. In the repo, unzip into results/final_telecom/ so
# results/final_telecom/outputs/val_forecasts.csv exists.
!rm -f val_outputs.zip
!zip val_outputs.zip outputs/val_forecasts.csv outputs/val_metrics.json
from google.colab import files
files.download("val_outputs.zip")